In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU не найдена!\n"
        "В Colab: Сменить среду выполнения → Графический процессор T4 → Сохранить.\n"
        "После этого перезапусти ячейки."
    )

print("✅ GPU:", torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/deni10000/MangaAutoTranslate

Cloning into 'MangaAutoTranslate'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 50 (delta 3), reused 50 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 7.73 MiB | 32.84 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [ ]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu125
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 603.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
%cd /content/MangaAutoTranslate
!pip install -r requirements.txt

/content/MangaAutoTranslate
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.1/175.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.2/272.2 k

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess
import time
import re
import sys
from threading import Thread

print("Запускаю Streamlit и Cloudflare...")

streamlit_proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_proc = subprocess.Popen([
    "./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None

def monitor_output(proc, prefix, find_url=False):
    global public_url
    for line in iter(proc.stdout.readline, ''):
        clean_line = line.strip()
        if clean_line:
            print(f"[{prefix}] {clean_line}")

        if find_url and not public_url:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", clean_line)
            if match:
                public_url = match.group(0)
                print("\n" + "="*60)
                print(f"ГОТОВО! Ссылка: {public_url}")
                print("="*60 + "\n")

Thread(target=monitor_output, args=(streamlit_proc, "STREAMLIT"), daemon=True).start()
Thread(target=monitor_output, args=(tunnel_proc, "CLOUDFLARE", True), daemon=True).start()

try:
  streamlit_proc.wait()
except KeyboardInterrupt:
    print("\nОстановка серверов...")
    streamlit_proc.terminate()
    tunnel_proc.terminate()

Запускаю Streamlit и Cloudflare...
[CLOUDFLARE] 2026-07-27T23:15:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[CLOUDFLARE] 2026-07-27T23:15:21Z INF Requesting new quick Tunnel on trycloudflare.com...
[STREAMLIT] Collecting usage statistics. To deactivate, set browser.gatherUsageStats to false.
[STREAMLIT] 2026-07-27 23:15:24.069 Uvicorn server started on :::8501
[STREAMLIT] You can now view your Streamlit app in your browser.
[STREAMLIT] Local URL: http://